In [1]:
import pygame
import sys

# Clase Personaje
class Personaje:
    def __init__(self, clase, agilidad, vitalidad):
        self.clase = clase
        self.agilidad = agilidad
        self.vitalidad = vitalidad

    def __repr__(self):
        return f"Personaje(clase={self.clase}, agilidad={self.agilidad}, vitalidad={self.vitalidad})"


def main():
    pygame.init()

    ANCHO, ALTO = 800, 600
    pantalla = pygame.display.set_mode((ANCHO, ALTO))
    pygame.display.set_caption("Menú principal - Grupo 5")

    BLANCO = (255, 255, 255)
    NEGRO = (0, 0, 0)
    AZUL = (0, 100, 255)
    AZUL_CLARO = (100, 150, 255)
    GRIS = (50, 50, 50)

    fuente_titulo = pygame.font.Font(None, 72)
    fuente_boton = pygame.font.Font(None, 48)
    fuente_texto = pygame.font.Font(None, 36)

    reloj = pygame.time.Clock()
    FPS = 60

    class Boton:
        def __init__(self, x, y, ancho, alto, texto):
            self.rect = pygame.Rect(x, y, ancho, alto)
            self.texto = texto
            self.color_base = AZUL
            self.color_hover = AZUL_CLARO
            self.color_actual = self.color_base

        def dibujar(self, superficie):
            pygame.draw.rect(superficie, self.color_actual, self.rect)
            pygame.draw.rect(superficie, BLANCO, self.rect, 3)
            texto_surface = fuente_boton.render(self.texto, True, BLANCO)
            texto_rect = texto_surface.get_rect(center=self.rect.center)
            superficie.blit(texto_surface, texto_rect)

        def actualizar(self, pos_mouse):
            if self.rect.collidepoint(pos_mouse):
                self.color_actual = self.color_hover
            else:
                self.color_actual = self.color_base

        def fue_clickeado(self, pos_mouse):
            return self.rect.collidepoint(pos_mouse)

    # ==================== MENÚ PRINCIPAL ====================
    # Botones: "Personaje" y "Salir del juego"
    boton_personaje = Boton(ANCHO // 2 - 150, ALTO // 2 - 50, 300, 80, "Personaje")
    boton_salir = Boton(ANCHO // 2 - 150, ALTO // 2 + 100, 300, 80, "Salir del juego")

    # ==================== PANTALLA A: 6 CUADRADOS CON "+" ====================
    # Layout de 6 cuadrados (3 columnas x 2 filas)
    cuadro_w, cuadro_h = 120, 120
    gap_x, gap_y = 30, 30
    cols = 3
    rows = 2
    total_cuadros = cols * rows

    cuadros_rects = []
    total_width = cols * cuadro_w + (cols - 1) * gap_x
    start_x = ANCHO // 2 - total_width // 2
    start_y = 140

    for r in range(rows):
        for c in range(cols):
            x = start_x + c * (cuadro_w + gap_x)
            y = start_y + r * (cuadro_h + gap_y)
            cuadros_rects.append(pygame.Rect(x, y, cuadro_w, cuadro_h))

    cuadro_hover_index = -1  # Índice del cuadro donde está el ratón (-1 si ninguno)

    boton_volver_pantalla_a = Boton(ANCHO // 2 - 150, ALTO - 120, 300, 60, "Volver atrás")

    # ==================== SELECCIÓN DE CLASE ====================
    btn_w, btn_h = 300, 60
    start_y_clases = ALTO // 2 - 150
    clases = [("Clérigo", 'clerigo'), ("Guerrero", 'guerrero'), ("Mago", 'mago'), ("Luchador", 'luchador')]
    botones_clases = []
    for i, (label, key) in enumerate(clases):
        y = start_y_clases + i * (btn_h + 10)
        botones_clases.append((key, Boton(ANCHO // 2 - btn_w // 2, y, btn_w, btn_h, label)))
    boton_volver_seleccion = Boton(ANCHO // 2 - btn_w // 2, start_y_clases + len(clases) * (btn_h + 10) + 20, btn_w, btn_h, "Volver atrás")

    # ==================== ESTADÍSTICAS BASE POR CLASE ====================
    stats = {'clerigo': (5, 12), 'guerrero': (7, 15), 'mago': (8, 8), 'luchador': (10, 10)}

    # ==================== ESTADO Y VARIABLES ====================
    ejecutando = True
    en_menu_principal = True
    en_pantalla_a = False
    en_seleccion_clase = False
    en_pantalla_clase = False
    personajes_creados = []
    MAX_PERSONAJES = 6
    clase_seleccionada = None  # Almacena la clase seleccionada para la pantalla de confirmación

    # ==================== BOTÓN CONFIRMAR EN PANTALLA DE CLASE ====================
    boton_confirmar_clase = Boton(ANCHO // 2 - 150, ALTO - 120, 300, 60, "Confirmar")
    boton_volver_pantalla_clase = Boton(ANCHO // 2 - 150, ALTO - 50, 300, 60, "Volver atrás")

    # ==================== BUCLE PRINCIPAL ====================
    while ejecutando:
        reloj.tick(FPS)
        pos_mouse = pygame.mouse.get_pos()

        # ==================== MANEJO DE EVENTOS ====================
        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                ejecutando = False

            if evento.type == pygame.MOUSEBUTTONDOWN and evento.button == 1:
                # --- Menú Principal ---
                if en_menu_principal:
                    if boton_personaje.fue_clickeado(pos_mouse):
                        if len(personajes_creados) < MAX_PERSONAJES:
                            en_menu_principal = False
                            en_pantalla_a = True
                    if boton_salir.fue_clickeado(pos_mouse):
                        ejecutando = False

                # --- Pantalla A (6 Cuadrados con +) ---
                elif en_pantalla_a:
                    # Click en uno de los cuadrados vacíos abre selección de clase
                    for i, cuadro in enumerate(cuadros_rects):
                        if i >= len(personajes_creados) and cuadro.collidepoint(pos_mouse):
                            # Solo permite crear si hay espacio
                            if len(personajes_creados) < MAX_PERSONAJES:
                                en_pantalla_a = False
                                en_seleccion_clase = True
                            break

                    # Botón "Volver atrás"
                    if boton_volver_pantalla_a.fue_clickeado(pos_mouse):
                        en_pantalla_a = False
                        en_menu_principal = True

                # --- Selección de Clase ---
                elif en_seleccion_clase:
                    # Seleccionar clase
                    for key, btn in botones_clases:
                        if btn.fue_clickeado(pos_mouse):
                            clase_seleccionada = key.capitalize()
                            en_seleccion_clase = False
                            en_pantalla_clase = True
                            break

                    # Botón "Volver atrás" regresa a pantalla A
                    if boton_volver_seleccion.fue_clickeado(pos_mouse):
                        en_seleccion_clase = False
                        en_pantalla_a = True

                # --- Pantalla de Clase (Muestra la clase seleccionada) ---
                elif en_pantalla_clase:
                    # Botón "Confirmar" crea el personaje
                    if boton_confirmar_clase.fue_clickeado(pos_mouse):
                        if clase_seleccionada and len(personajes_creados) < MAX_PERSONAJES:
                            agi, vit = stats[clase_seleccionada.lower()]
                            nuevo = Personaje(clase_seleccionada, agi, vit)
                            personajes_creados.append(nuevo)
                        en_pantalla_clase = False
                        en_pantalla_a = True
                        clase_seleccionada = None

                    # Botón "Volver atrás" regresa a selección de clase
                    if boton_volver_pantalla_clase.fue_clickeado(pos_mouse):
                        en_pantalla_clase = False
                        en_seleccion_clase = True
                        clase_seleccionada = None

        # ==================== ACTUALIZACIÓN DE BOTONES (HOVER) ====================
        if en_menu_principal:
            boton_personaje.actualizar(pos_mouse)
            boton_salir.actualizar(pos_mouse)
        elif en_pantalla_a:
            boton_volver_pantalla_a.actualizar(pos_mouse)
            # Detectar hover en cuadros
            cuadro_hover_index = -1
            for i, cuadro in enumerate(cuadros_rects):
                if i >= len(personajes_creados) and cuadro.collidepoint(pos_mouse):
                    cuadro_hover_index = i
                    break
        elif en_seleccion_clase:
            for _, btn in botones_clases:
                btn.actualizar(pos_mouse)
            boton_volver_seleccion.actualizar(pos_mouse)
        elif en_pantalla_clase:
            boton_confirmar_clase.actualizar(pos_mouse)
            boton_volver_pantalla_clase.actualizar(pos_mouse)

        # ==================== DIBUJO DE PANTALLA ====================
        pantalla.fill(NEGRO)

        # --- MENÚ PRINCIPAL ---
        if en_menu_principal:
            titulo = fuente_titulo.render("Menú Principal", True, BLANCO)
            titulo_rect = titulo.get_rect(center=(ANCHO // 2, 100))
            pantalla.blit(titulo, titulo_rect)

            boton_personaje.dibujar(pantalla)
            boton_salir.dibujar(pantalla)

        # --- PANTALLA A (6 Cuadrados con +) ---
        elif en_pantalla_a:
            titulo = fuente_titulo.render("Crear Personaje", True, BLANCO)
            titulo_rect = titulo.get_rect(center=(ANCHO // 2, 80))
            pantalla.blit(titulo, titulo_rect)

            # Dibujar los 6 cuadrados
            for i, cuadro in enumerate(cuadros_rects):
                if i < len(personajes_creados):
                    # Cuadro ocupado: mostrar personaje
                    color_cuadro = AZUL
                    pygame.draw.rect(pantalla, color_cuadro, cuadro)
                    pygame.draw.rect(pantalla, BLANCO, cuadro, 3)
                    # Mostrar inicial de la clase
                    inicial = fuente_boton.render(personajes_creados[i].clase[0], True, BLANCO)
                    pantalla.blit(inicial, inicial.get_rect(center=cuadro.center))
                else:
                    # Cuadro vacío: mostrar "+"
                    color_cuadro = AZUL_CLARO if cuadro_hover_index == i else AZUL
                    pygame.draw.rect(pantalla, color_cuadro, cuadro)
                    pygame.draw.rect(pantalla, BLANCO, cuadro, 3)
                    plus = fuente_titulo.render("+", True, BLANCO)
                    pantalla.blit(plus, plus.get_rect(center=cuadro.center))

            # Botón "Volver atrás"
            boton_volver_pantalla_a.dibujar(pantalla)

            # Contador de personajes
            contador = fuente_texto.render(f"Personajes: {len(personajes_creados)}/{MAX_PERSONAJES}", True, BLANCO)
            pantalla.blit(contador, (20, 20))

        # --- SELECCIÓN DE CLASE ---
        elif en_seleccion_clase:
            titulo = fuente_titulo.render("Escoger clase", True, BLANCO)
            titulo_rect = titulo.get_rect(center=(ANCHO // 2, 80))
            pantalla.blit(titulo, titulo_rect)

            for _, btn in botones_clases:
                btn.dibujar(pantalla)

            boton_volver_seleccion.dibujar(pantalla)

        # --- PANTALLA DE CLASE (Título con la clase seleccionada) ---
        elif en_pantalla_clase and clase_seleccionada:
            titulo = fuente_titulo.render(clase_seleccionada, True, BLANCO)
            titulo_rect = titulo.get_rect(center=(ANCHO // 2, 150))
            pantalla.blit(titulo, titulo_rect)

            # Mostrar estadísticas de la clase
            agi, vit = stats[clase_seleccionada.lower()]
            stats_text = fuente_texto.render(f"Agilidad: {agi}   Vitalidad: {vit}", True, BLANCO)
            pantalla.blit(stats_text, stats_text.get_rect(center=(ANCHO // 2, 280)))

            # Botones confirmar y volver
            boton_confirmar_clase.dibujar(pantalla)
            boton_volver_pantalla_clase.dibujar(pantalla)

        # ==================== ACTUALIZACIÓN DE PANTALLA ====================
        pygame.display.flip()

    # ==================== CIERRE DE PYGAME ====================
    pygame.quit()
    sys.exit()


if __name__ == "__main__":
    main()


pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


SystemExit: 

c:\Users\valen\Documents\GitHub\juegoRPG\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
